# Week 08 — Home exercise 3: Fifty-five years of the NASDAQ

**Solution proposal.**

Dates, downsampling, upsampling, and what fifty-five years of a stock index looks like when you make
it answer questions.

In [1]:
import pandas as pd

nasdaq = pd.read_csv("../data/NASDAQ.csv")
nasdaq["Date"] = pd.to_datetime(nasdaq["Date"], format="%Y-%m-%d")

prices = nasdaq.set_index("Date").sort_index()["NASDAQ"]

print(prices.dtype, "|", len(prices), "trading days")

float64 | 13842 trading days


## 1. Range, and trading days per decade

In [2]:
print("first:", prices.index.min().date())
print("last: ", prices.index.max().date())
print()
print(prices.groupby(prices.index.year // 10 * 10).size())

first: 1971-02-05
last:  2025-12-30

Date
1970    2248
1980    2528
1990    2528
2000    2515
2010    2516
2020    1507
Name: NASDAQ, dtype: int64


Five full decades of roughly 2 520 trading days each — about 252 a year, which is 365 minus weekends
and public holidays. The 1970s are short because the series starts in February 1971, and the 2020s are
short because they are not finished.

That regularity is worth noticing before doing anything else: it tells you the file has no missing
stretches, so any gap you find later is a weekend or a holiday rather than lost data.

## 2. Annual closing levels and yearly returns

In [3]:
annual = prices.resample("YE").last()
annual_return = annual.pct_change() * 100

print("worst five years:")
print(annual_return.sort_values().head(5).round(1))
print()
print("best five years:")
print(annual_return.sort_values(ascending=False).head(5).round(1))

worst five years:
Date
2008-12-31   -40.5
2000-12-31   -39.3
1974-12-31   -35.1
2022-12-31   -33.1
2002-12-31   -31.5
Name: NASDAQ, dtype: float64

best five years:
Date
1999-12-31    85.6
1991-12-31    56.8
2003-12-31    50.0
2009-12-31    43.9
2020-12-31    43.6
Name: NASDAQ, dtype: float64


**The worst two are 2008 and 2000.**

- **2008, down 40.5%** — the global financial crisis. Lehman Brothers failed in September and the
  whole market fell with it.
- **2000, down 39.3%** — the dot-com crash. The NASDAQ is heavily weighted towards technology
  companies, so it was hit far harder than the broader market when the internet bubble burst.

And notice **1999, up 85.6%** — the year immediately before. The best year in the series and the second
worst are consecutive, which is the single most useful thing this dataset teaches.

## 3. The three worst months

In [4]:
monthly = prices.resample("ME").last()
monthly_return = monthly.pct_change() * 100

monthly_return.sort_values().head(3).round(1)

Date
1987-10-31   -27.2
2000-11-30   -22.9
2001-02-28   -22.4
Name: NASDAQ, dtype: float64

**October 1987, down 27.2%** — "Black Monday", 19 October 1987, when markets around the world fell
more than 20% in a single day. It remains the largest one-day percentage fall in modern stock market
history, and nearly forty years later it is still the worst month in this file.

The other two, November 2000 and February 2001, are the dot-com crash again, this time visible month
by month rather than as one annual number.

## 4. Upsampling to every calendar day

In [5]:
daily = prices.resample("D").last()

print("trading days:   ", len(prices))
print("calendar days:  ", len(daily))
print("of which empty: ", daily.isna().sum(), f"({daily.isna().sum() / len(daily) * 100:.0f}%)")
print()
print(daily.loc["2020-03-05":"2020-03-10"].round(1))

trading days:    13842
calendar days:   20053
of which empty:  6211 (31%)

Date
2020-03-05    8738.6
2020-03-06    8575.6
2020-03-07       NaN
2020-03-08       NaN
2020-03-09    7950.7
2020-03-10    8344.2
Freq: D, Name: NASDAQ, dtype: float64


20 053 calendar days, **6 211 of them empty** — 31%, which is roughly two days in seven plus public
holidays. Those are exactly the weekends: Saturday 7 and Sunday 8 March 2020 are `NaN` because no
trading happened.

The rows did not exist before. `resample` built the periods from the calendar, so days with no
observation appear as gaps instead of quietly not being there.

In [6]:
filled = daily.ffill()

print("empty after ffill:", filled.isna().sum())
filled.loc["2020-03-05":"2020-03-10"].round(1)

empty after ffill: 0


Date
2020-03-05    8738.6
2020-03-06    8575.6
2020-03-07    8575.6
2020-03-08    8575.6
2020-03-09    7950.7
2020-03-10    8344.2
Freq: D, Name: NASDAQ, dtype: float64

### When is carrying the last price forward reasonable?

**Reasonable** when the underlying quantity genuinely did not change, and you need a value for every
calendar day. A market is *closed* on Saturday — there is no price because nothing traded, not because
the observation was lost. Saying Saturday's level equals Friday's is close to a statement of fact, and
it is what you want if you are lining this series up against something recorded seven days a week.

**Not reasonable** when the value *did* change and you simply do not know what it became. Carrying a
country's last published statistic forward across three years, as we did with renewable energy in the
lecture, invents a flat line for exactly the period everyone wants to look at. Same method, same one
line of code, completely different honesty.

The test is not technical: **would the number have moved while you were not looking?** If yes, filling
is a guess wearing the costume of data.

## 5. `groupby` on the month, against `resample`

In [7]:
by_month = prices.groupby(prices.index.month).mean()
by_period = prices.resample("ME").mean()

print("groupby rows: ", len(by_month))
print("resample rows:", len(by_period))
print()
print(by_month.round(0).head(4))

groupby rows:  12
resample rows: 659

Date
1    2954.0
2    3028.0
3    2955.0
4    2967.0
Name: NASDAQ, dtype: float64


**Twelve rows against 659, and they measure different things.**

`groupby` on `.dt.month` puts every January in the file into one group. `.dt.month` returns the number
1 for January 1971 and for January 2025 alike, so the "January" figure of 2 954 is an average across
fifty-five Januaries during which the index went from 100 to 23 419. It is not the level in any
January that ever happened.

`resample("ME")` keeps the periods separate and dated: one row for January 1971, another for January
2025, 659 in all.

Which is right depends on the question:

- **`groupby` on a date part** is the tool for *seasonal* questions — "are Januaries usually weak?" —
  where merging fifty-five Januaries is exactly the point. Even then, on a series that grows by a
  factor of 234 you would want to compare *returns* rather than levels, or the later years would
  drown the earlier ones.
- **`resample`** is the tool for questions about a series *through time*.

**Only `resample` belongs on a chart with time along the bottom**, because only it has a real date on
every row. Plotting the `groupby` result would put the numbers 1 to 12 on the axis, and a reader would
have no way to know they span half a century.

## The lost decade

In [8]:
end_1999 = annual.loc["1999-12-31"]
end_2009 = annual.loc["2009-12-31"]

print(f"end of 1999: {end_1999:8.1f}")
print(f"end of 2009: {end_2009:8.1f}")
print(f"change:      {(end_2009 / end_1999 - 1) * 100:8.1f}%")

later = annual[annual.index.year > 1999]
recovered = later[later > end_1999]

print()
print("first year closing above the 1999 level:", recovered.index[0].year)

end of 1999:   4069.3
end of 2009:   2269.1
change:         -44.2%

first year closing above the 1999 level: 2013


**The NASDAQ ended 2009 forty-four percent below where it ended 1999, and did not close above its 1999
level again until 2013.**

That is thirteen years to get back to level in **nominal** terms, and prices did not stand still
meanwhile: `FRED_annual.xlsx`, which is in the same data folder, puts US consumer prices 39.9% higher
in 2013 than in 1999. So an investor who bought at the end of 1999 and sold on the day the index
finally got back to level had lost nearly a third of their purchasing power.

### What that does to "shares go up over time"

The claim is not false, and it is not a promise about any particular decade. Over the full 55 years
here the index went from 100 to 23 419, which is an enormous return. But it got there through one
period where a third of its value vanished in a year and did not come back for over a decade — and
somebody who needed the money in 2009 got the decade, not the average.

Two honest caveats on this specific evidence:

- **The NASDAQ is not the market.** It is unusually concentrated in technology companies, which is
  exactly why the dot-com crash hit it so hard. A broader index fell much less over the same decade.
- **An index is not an investment.** It ignores dividends, which add meaningfully to long-run returns,
  and it ignores the fees and taxes of actually holding anything.

### Things worth noticing

- **`.set_index("Date").sort_index()` before anything else.** `resample` on an unsorted index does not
  raise; it produces something wrong.
- **`resample` then `.pct_change()`** is the whole recipe for a return series, and it works at any
  frequency — annual, monthly, or daily — because both methods leave you with a dated Series.
- `prices.index.year // 10 * 10` groups by decade without creating a column. Grouping by something you
  compute on the spot is allowed and often tidier.

### What this notebook does NOT do

- **It never asks whether the file is complete.** 2 528 trading days in the 1980s looks right, but
  nothing here would catch a fortnight missing from 1994 — and after `resample("YE")` a year with half
  its days missing produces a number that looks exactly like every other year.
- **The returns are index returns**, not investor returns: no dividends, no fees, no tax, and no
  account of the fact that the composition of the index changed completely over 55 years.
- **`.last()` takes the final trading day of each period**, so an annual figure is one day's closing
  level rather than anything averaged. That is the convention for a price series, and it does mean a
  single unusual day at the end of December decides the whole year's number.